# DreamerV4 CUDA Training

This notebook trains the native PyTorch DreamerV4 agent on the blood-supply simulator with CUDA, publishes a drop-in `dreamerv4_agent.pt`, and validates it in the current simulator runtime.

In [ ]:
from pathlib import Path
import json
import sys


def find_ml_backend_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        direct = candidate / "simulator" / "dreamerv4_notebook.py"
        nested = candidate / "ml-backend" / "simulator" / "dreamerv4_notebook.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "ml-backend"
    raise FileNotFoundError(
        "Could not locate ml-backend/simulator from the current working directory."
    )


ML_BACKEND_DIR = find_ml_backend_dir(Path.cwd().resolve())
SIMULATOR_DIR = ML_BACKEND_DIR / "simulator"
if str(SIMULATOR_DIR) not in sys.path:
    sys.path.insert(0, str(SIMULATOR_DIR))

from dreamerv4_notebook import (
    DEFAULT_DROPIN_EXPORT_DIR,
    DEFAULT_NOTEBOOK_LOGDIR,
    DreamerV4NotebookConfig,
    evaluate_checkpoint,
    load_metrics_frame,
    runtime_report,
    train_from_notebook,
)

print(f"ml-backend: {ML_BACKEND_DIR}")
print(f"simulator:  {SIMULATOR_DIR}")
print(f"runs dir:   {DEFAULT_NOTEBOOK_LOGDIR}")
print(f"drop-in:    {DEFAULT_DROPIN_EXPORT_DIR}")


In [ ]:
report = runtime_report("cuda")
print(json.dumps(report, indent=2))

if not report["cuda_available"]:
    raise RuntimeError(
        "CUDA is not available on this machine. Use a CUDA host or change the notebook config to device='auto'."
    )


In [ ]:
config = DreamerV4NotebookConfig(
    steps=50_000,
    seed=7,
    logdir=str(DEFAULT_NOTEBOOK_LOGDIR),
    device="cuda",
    model_preset="fast",
    envs=0,
    batch_size=32,
    seq_len=48,
    imagine_horizon=15,
    train_ratio=1.0,
    publish_dropin=True,
    export_dir=str(DEFAULT_DROPIN_EXPORT_DIR),
)

config


In [ ]:
train_result = train_from_notebook(config)
train_result


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

metrics_df = load_metrics_frame(train_result["logdir"])
display(metrics_df.tail())

plot_df = metrics_df.dropna(subset=["step"]).copy()
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

episode_df = metrics_df.dropna(subset=["episode_reward"]) if "episode_reward" in metrics_df else metrics_df.iloc[0:0]
if not episode_df.empty:
    episode_df.plot(x="step", y="episode_reward", ax=axes[0], title="Episode Reward")
else:
    axes[0].set_title("Episode Reward")

if "wm/total" in plot_df and not plot_df.empty:
    plot_df.plot(x="step", y="wm/total", ax=axes[1], title="World Model Loss")
else:
    axes[1].set_title("World Model Loss")

if "ac/critic_loss" in plot_df and not plot_df.empty:
    plot_df.plot(x="step", y="ac/critic_loss", ax=axes[2], title="Critic Loss")
else:
    axes[2].set_title("Critic Loss")

plt.tight_layout()


In [ ]:
eval_result = evaluate_checkpoint(
    train_result["dropin_checkpoint"],
    scenario_key="baseline",
    hours=72,
    seed=config.seed,
    fast_mode=True,
)

eval_result


## Using The Model In The Current Simulator

By default this notebook publishes the trained checkpoint into the simulator's runtime drop-in location:

`ml-backend/simulator/dreamerv4_runs/native/ckpt/dreamerv4_agent.pt`

That means the current `dreamerv4` strategy can load it without any simulator code changes. If you move the checkpoint between machines, copy the single `dreamerv4_agent.pt` file into any simulator `ckpt/` directory, or point `PIOS_DREAMERV4_CHECKPOINT` at the file or its checkpoint directory.